# 03c · Anotación en CVAT: zonas por cámara y estado de los vehículos

## Motivación / Objetivos

Cerca de muchos semáforos hay coches aparcados. Si se cuentan como "vehículos esperando", los indicadores de cola y de espera de `05` salen inflados. En una sola imagen, un coche aparcado y uno detenido en rojo son casi indistinguibles, así que `06_clasificacion_estado_vehiculos` combina zonas fijas por cámara con la persistencia temporal.

Este notebook reúne la **anotación manual** que necesita esa clasificación:
- **Zonas por cámara.** Para cada cámara de referencia se dibujan una vez en CVAT polígonos de `zona_espera` (carril de espera hasta la línea de detención), `zona_aparcamiento` (plazas visibles) y `paso_peatones` (paso de cebra).
- **Lote de validación.** Imágenes de las secuencias a 6 s en las que se marca el atributo `estado` de cada coche (`desconocido`, `esperando`, `aparcado`, `circulando`). Es la referencia manual para medir cuánto aciertan las reglas.

**Objetivos del TFM relacionados:** OE-5 (relación entre peatones, flujo vehicular y semáforos: la cola real excluye los vehículos aparcados).

## Método

Parámetros en `config.yaml` → `estado_vehiculos`. Lógica de CVAT en `src/cvat_lotes.py`.

1. Configuración.
2. Tarea de zonas en CVAT.
3. Exportación de las zonas.
4. Comprobación visual de las zonas.
5. Lote de validación del estado de los vehículos.
6. Exportación del lote de validación.

### 1. Configuración

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Polygon

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import PROJECT_ROOT as ROOT, load_config, path
from cvat_lotes import (connect, create_lote, create_zones_task, export_lote, export_zones, get_or_create_project,
                        get_or_create_zones_project, images_in_lotes, load_task, lote_dir, pre_annotate, task_url)
from estado_vehiculos import continuous_sequences, list_reference_images, load_zones

CFG = load_config()
EV = CFG["estado_vehiculos"]
CAMERAS = [str(c) for c in EV["camaras"]]
ZONES_FILE = ROOT / EV["zonas"]
ZONES_TASK_FILE = ZONES_FILE.parent / "cvat_task.json"

print("Cámaras de referencia:", CAMERAS)
print("Fichero de zonas:", ZONES_FILE)

### 2. Tarea de zonas en CVAT

**Primera vez:** la celda crea en CVAT el proyecto *"TFM Malaga - zonas de espera"* y una tarea con una imagen diurna nítida de cada cámara de referencia. En cada imagen hay que dibujar con la herramienta de **polígono**:
- `zona_espera`: el carril o carriles donde los coches se detienen ante el semáforo, hasta la línea de detención.
- `zona_aparcamiento`: las plazas de aparcamiento (en batería o en línea) cercanas.
- `paso_peatones`: el paso de cebra. Un vehículo detenido sobre él no cuenta como "esperando". La zona servirá también para inferir la fase peatonal a partir del semáforo vehicular de la calle que cruzan los peatones.

Puede haber varios polígonos de cada tipo por cámara. Después se marca el job como terminado.

In [2]:
client = connect()

if ZONES_TASK_FILE.exists():
    zones_task = client.tasks.retrieve(json.loads(ZONES_TASK_FILE.read_text())["task_id"])
    print(f"Tarea de zonas existente: {zones_task.id}")
else:
    frames = pd.read_parquet(path("fases") / "frames.parquet")
    frames["camera"] = frames["camera"].astype(str)
    # Una imagen de día, la de mejor calidad de cada cámara de referencia
    best = (frames[frames["camera"].isin(CAMERAS) & (frames["illumination"] == "dia")]
            .sort_values("quality_score", ascending=False).groupby("camera").head(1))
    ref_dir = ZONES_FILE.parent / "imagenes"
    ref_dir.mkdir(parents=True, exist_ok=True)
    images = []
    for row in best.itertuples():
        target = ref_dir / f"{row.camera}_{row.image}"
        target.write_bytes((ROOT / row.path).read_bytes())
        images.append(target)
    zones_task = create_zones_task(client, get_or_create_zones_project(client), images)
    ZONES_TASK_FILE.write_text(json.dumps({"task_id": zones_task.id}, indent=2))
    print(f"Tarea de zonas creada: {zones_task.id} ({len(images)} cámaras)")

print(f"Dibujar las zonas en: {task_url(zones_task)}")

Tarea de zonas existente: 3
Dibujar las zonas en: http://localhost:8080/tasks/3


### 3. Exportación de las zonas

Guarda los polígonos en `data/malaga/zonas/zonas_camaras.json` (versionado en git), con puntos normalizados (0–1).

In [3]:
zones = export_zones(client, zones_task, ZONES_FILE)
resumen = {cam: {z: len(p) for z, p in zs.items()} for cam, zs in zones.items()}
print(f"Zonas guardadas en {ZONES_FILE}:")
display(pd.DataFrame(resumen).T if resumen else "Aún no hay polígonos dibujados en CVAT.")

Zonas guardadas en C:\Users\loren\OneDrive\Escritorio\MASTER-UIMP\TFM\tfm\data\malaga\zonas\zonas_camaras.json:


'Aún no hay polígonos dibujados en CVAT.'

### 4. Comprobación visual de las zonas

In [4]:
# Comprobación visual de las zonas sobre la imagen de referencia de cada cámara
zones = load_zones(ZONES_FILE)
ref_images = {p.stem.split("_", 1)[0]: p for p in (ZONES_FILE.parent / "imagenes").glob("*.jpg")}
if zones:
    fig, axes = plt.subplots(1, len(zones), figsize=(5 * len(zones), 4), squeeze=False)
    for ax, (cam, cam_zones) in zip(axes[0], zones.items()):
        img = cv2.cvtColor(cv2.imread(str(ref_images[cam])), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        ax.imshow(img)
        for zone, color in [("zona_espera", "lime"), ("zona_aparcamiento", "red"), ("paso_peatones", "deepskyblue")]:
            for poly in cam_zones.get(zone, []):
                ax.add_patch(Polygon([(x * w, y * h) for x, y in poly], fill=False, edgecolor=color, lw=2, label=zone))
        ax.set_title(cam)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Sin zonas: la clasificación usará solo la persistencia temporal (ningún vehículo saldrá como 'esperando').")

Sin zonas: la clasificación usará solo la persistencia temporal (ningún vehículo saldrá como 'esperando').


### 5. Lote de validación del estado de los vehículos

Crea el lote `lote_03_estado_vehiculos` con `N_POR_CAMARA` imágenes de cada cámara de referencia, repartidas a lo largo de sus secuencias a 6 s (no consecutivas) y pre-anotadas con YOLO26. En CVAT se revisan las cajas con el protocolo de `03a` y se marca el atributo `estado` de cada coche. Se ejecuta una sola vez (`CREAR_LOTE_VALIDACION = True`).

In [ ]:
CREAR_LOTE_VALIDACION = False   # True para crear el lote en CVAT (una sola vez)
LOTE_VALIDACION = "lote_03_estado_vehiculos"
N_POR_CAMARA = 10

seq = continuous_sequences(list_reference_images(CAMERAS), EV["max_hueco_s"])
print(f"Imágenes en secuencias de ≤ {EV['max_hueco_s']} s: {len(seq)}")

if (lote_dir(LOTE_VALIDACION) / "indice.csv").exists():
    print(f"El lote {LOTE_VALIDACION} ya existe: {task_url(load_task(client, LOTE_VALIDACION))}")
elif CREAR_LOTE_VALIDACION:
    used = images_in_lotes()
    cand = seq.assign(image_lote=seq["camera"] + "_" + seq["image"])
    cand = cand[~cand["image_lote"].isin(used)]
    # Imágenes repartidas a lo largo de cada secuencia (no consecutivas)
    pick = cand.groupby("camera", group_keys=False).apply(
        lambda g: g.iloc[::max(1, len(g) // N_POR_CAMARA)].head(N_POR_CAMARA))
    selection = pick.assign(image=pick["image_lote"])
    project = get_or_create_project(client)
    task = create_lote(client, project, LOTE_VALIDACION, selection, segment_size=len(selection))
    pre_annotate(client, task, CFG["deteccion"]["modelo"], CFG["deteccion"]["conf"], CFG["deteccion"]["imgsz"])
    print(f"Lote de validación creado: {task_url(task)}. Marca el atributo 'estado' de cada coche.")

### 6. Exportación del lote de validación

Igual que en `03a`: solo pasan a `revisada` las imágenes de jobs terminados. `06` lee el atributo `estado` directamente de CVAT.

In [ ]:
if (lote_dir(LOTE_VALIDACION) / "indice.csv").exists():
    indice = export_lote(client, load_task(client, LOTE_VALIDACION), LOTE_VALIDACION)
    print(indice["estado"].value_counts().to_dict())

## Salida / Resultados

| Salida | Contenido |
|---|---|
| `data/malaga/zonas/zonas_camaras.json` | Polígonos por cámara (versionado). |
| `data/malaga/zonas/cvat_task.json` | Identificador de la tarea de zonas en CVAT. |
| `data/malaga/zonas/imagenes/` | Imagen de referencia de cada cámara (no versionada). |
| `data/malaga/anotacion/lote_03_estado_vehiculos/` | Lote de validación, con la misma estructura que los de `03a`. |

**Estado (17/10/2026):** la tarea de zonas (tarea 3) está creada con las 5 cámaras de referencia, pero aún no tiene polígonos. El lote de validación no se ha creado.

## Observaciones

- **Por qué está separado de `06`:** las celdas que crean tareas en CVAT y exportan anotaciones humanas pertenecen a la fase de anotación. Seguimiento y clasificación son análisis: consumen las zonas exportadas y las detecciones a 6 s, y se vuelven a ejecutar cada vez que llegan imágenes nuevas, sin tocar CVAT.
- Hay que dibujar las zonas **antes** de ejecutar `06`. Sin ellas, ningún vehículo sale como "esperando".
- El lote de validación es también un lote de entrenamiento más: `04` lee todos los `anotacion/*/indice.csv` con imágenes revisadas.
- **Pendiente:** implementar la inferencia de la fase peatonal a partir de `paso_peatones` y del semáforo vehicular de la calle que cruzan los peatones.